# 02 - Baseline LiDAR model on nuScenes

## Goal

This notebook documents the setup of a baseline LiDAR-only 3D object detector on nuScenes using MMDetection3D.

The objective is to train a clean baseline model that can later be compared against more advanced approaches such as fusion models or imbalance-mitigation variants.

## Experimental setting

The baseline is trained on:
- the nuScenes dataset prepared in the previous notebook
- a reproducible 20% training-scene subset
- the standard validation split

## Why this baseline matters

A LiDAR-only baseline is important because:
- it provides a reference point for later comparisons
- it establishes whether the training pipeline is working correctly
- it helps separate baseline performance from improvements due to fusion or other methods

## Planned outcome

This notebook should:
1. verify the MMDetection3D environment
2. identify the baseline config to use
3. confirm the subset annotation file paths
4. create a baseline training config for the reduced dataset
5. document the training command
6. prepare the experiment for reproducible execution

# Defining project paths

This notebook uses the same project and dataset paths as the dataset-preparation notebook.

I define them explicitly here so that the notebook is self-contained and can be run independently.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning")
MMDET3D_ROOT = PROJECT_ROOT / "external" / "mmdetection3d"
NUSCENES_ROOT = Path("/rs_scratch/users/ae04q066/nuscenes_full")
MMDET3D_DATA_ROOT = MMDET3D_ROOT / "data" / "nuscenes"

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("MMDET3D_ROOT     :", MMDET3D_ROOT)
print("NUSCENES_ROOT    :", NUSCENES_ROOT)
print("MMDET3D_DATA_ROOT:", MMDET3D_DATA_ROOT)

PROJECT_ROOT     : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
MMDET3D_ROOT     : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d
NUSCENES_ROOT    : /rs_scratch/users/ae04q066/nuscenes_full
MMDET3D_DATA_ROOT: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes


# Verifying the MMDetection3D environment

Before configuring the baseline model, I verify that:
- MMDetection3D is accessible
- the correct Python environment is active
- core libraries are properly installed

This ensures that training will run without unexpected environment issues.

In [2]:
import mmengine
import mmdet
import mmdet3d

print("mmengine version:", mmengine.__version__)
print("mmdet version   :", mmdet.__version__)
print("mmdet3d version :", mmdet3d.__version__)

mmengine version: 0.10.7
mmdet version   : 3.2.0
mmdet3d version : 1.4.0


## Environment verification result

The MMDetection3D environment is correctly set up.

Library versions:
- mmengine: 0.10.7
- mmdet: 3.2.0
- mmdet3d: 1.4.0

These versions are compatible and confirm that the training pipeline can be executed.

# Selecting the baseline model

For the LiDAR baseline, I use the **CenterPoint** detector, which is a standard and strong baseline for nuScenes 3D object detection.

### Why CenterPoint
- widely used benchmark model for nuScenes
- strong performance with LiDAR-only input
- fully supported in MMDetection3D

### Configuration choice

I use the official MMDetection3D config:

`centerpoint_voxel01_second_secfpn_nus.py`

This config defines:
- voxelization-based LiDAR processing
- SECOND backbone
- SECFPN neck
- CenterPoint detection head

# Baseline config location and safety rule

The CenterPoint nuScenes baseline config is available in the repository under the `mmdet3d/configs/centerpoint/` directory.

For this project, I do **not** modify the repository config in place.

Instead, I create a separate experiment config in my own project area. This is safer because:
- it preserves the original MMDetection3D configuration
- it avoids accidental overwrites
- it makes the experiment easier to document and reproduce

# Defining source and experiment config paths

I now define:
- the source CenterPoint baseline config from the MMDetection3D repository
- a dedicated experiment config folder in my project area

The experiment config will be created as a copy of the source config and then adapted for the reduced training subset.

In [3]:
source_config_path = (
    MMDET3D_ROOT
    / "configs"
    / "centerpoint"
    / "centerpoint_voxel01_second_secfpn_8xb4-cyclic-20e_nus-3d.py"
)

experiment_config_dir = MMDET3D_ROOT / "configs" / "my_experiments"
experiment_config_dir.mkdir(parents=True, exist_ok=True)

experiment_config_path = experiment_config_dir / "lidar_baseline_nuscenes_20pct.py"

print("Source config path     :", source_config_path)
print("Source exists          :", source_config_path.exists())
print("Experiment config dir  :", experiment_config_dir)
print("Experiment config path :", experiment_config_path)

Source config path     : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/centerpoint/centerpoint_voxel01_second_secfpn_8xb4-cyclic-20e_nus-3d.py
Source exists          : True
Experiment config dir  : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments
Experiment config path : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/lidar_baseline_nuscenes_20pct.py


# Copying the baseline config into the experiment folder

I now create a project-specific copy of the baseline CenterPoint config.

This copied config will be the one modified for the reduced dataset experiment.
The original MMDetection3D config remains untouched.

In [4]:
import shutil

if experiment_config_path.exists():
    print("Experiment config already exists:")
    print(experiment_config_path)
    print("\nSafety rule: existing file is not overwritten automatically.")
else:
    shutil.copy2(source_config_path, experiment_config_path)
    print("Copied source config to:")
    print(experiment_config_path)

print("\nExperiment config exists:", experiment_config_path.exists())

Experiment config already exists:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/lidar_baseline_nuscenes_20pct.py

Safety rule: existing file is not overwritten automatically.

Experiment config exists: True


# Inspecting the experiment config

Before modifying the config, I inspect its current content.

This allows me to:
- confirm it is based on the CenterPoint nuScenes baseline
- identify where dataset paths are defined
- locate the sections that need to be updated for the subset experiment

In [5]:
with open(experiment_config_path, "r") as f:
    lines = f.readlines()

print("First 60 lines of the config:\n")
for i, line in enumerate(lines[:60]):
    print(f"{i+1:03d}: {line.rstrip()}")

First 60 lines of the config:

001: _base_ = [
002:     '../_base_/datasets/nus-3d.py',
003:     '../_base_/models/centerpoint_voxel01_second_secfpn_nus.py',
004:     '../_base_/schedules/cyclic-20e.py', '../_base_/default_runtime.py'
005: ]
006: 
007: # If point cloud range is changed, the models should also change their point
008: # cloud range accordingly
009: point_cloud_range = [-51.2, -51.2, -5.0, 51.2, 51.2, 3.0]
010: # Using calibration info convert the Lidar-coordinate point cloud range to the
011: # ego-coordinate point cloud range could bring a little promotion in nuScenes.
012: # point_cloud_range = [-51.2, -52, -5.0, 51.2, 50.4, 3.0]
013: # For nuScenes we usually do 10-class detection
014: class_names = [
015:     'car', 'truck', 'construction_vehicle', 'bus', 'trailer', 'barrier',
016:     'motorcycle', 'bicycle', 'pedestrian', 'traffic_cone'
017: ]
018: data_prefix = dict(pts='samples/LIDAR_TOP', img='', sweeps='sweeps/LIDAR_TOP')
019: model = dict(
020:     data_prepro

# Locating dataset annotation settings

The baseline config inherits most settings from MMDetection3D base files.

I now search the experiment config for the dataset and dataloader sections that define:
- training annotation file
- validation annotation file
- test annotation file

These are the key fields that must point to the reduced training subset while keeping validation unchanged.

In [6]:
keywords = [
    "train_dataloader",
    "val_dataloader",
    "test_dataloader",
    "ann_file",
    "data_root",
    "dataset=",
]

for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        print(f"{i+1:03d}: {line.rstrip()}")

028: data_root = 'data/nuscenes/'
032:     data_root=data_root,
033:     info_path=data_root + 'nuscenes_dbinfos_train.pkl',
134: train_dataloader = dict(
140:     dataset=dict(
142:         dataset=dict(
144:             data_root=data_root,
145:             ann_file='subsets/nuscenes_infos_train_20pct.pkl',
155: test_dataloader = dict(
156:     dataset=dict(pipeline=test_pipeline, metainfo=dict(classes=class_names)))
157: val_dataloader = dict(
158:     dataset=dict(pipeline=test_pipeline, metainfo=dict(classes=class_names)))


# Inspecting the dataloader block

Before making any changes, I inspect the full dataloader section around the training and validation dataset definitions.

This makes it easier to:
- edit the correct fields safely
- understand whether the config uses wrappers such as `CBGSDataset`
- confirm exactly where `ann_file` is defined

In [7]:
start_line = 130
end_line = 170

for i, line in enumerate(lines[start_line - 1:end_line], start=start_line):
    print(f"{i:03d}: {line.rstrip()}")

130:         ]),
131:     dict(type='Pack3DDetInputs', keys=['points'])
132: ]
133: 
134: train_dataloader = dict(
135:     _delete_=True,
136:     batch_size=4,
137:     num_workers=16,
138:     persistent_workers=True,
139:     sampler=dict(type='DefaultSampler', shuffle=True),
140:     dataset=dict(
141:         type='CBGSDataset',
142:         dataset=dict(
143:             type=dataset_type,
144:             data_root=data_root,
145:             ann_file='subsets/nuscenes_infos_train_20pct.pkl',
146:             pipeline=train_pipeline,
147:             metainfo=dict(classes=class_names),
148:             test_mode=False,
149:             data_prefix=data_prefix,
150:             use_valid_flag=True,
151:             # we use box_type_3d='LiDAR' in kitti and nuscenes dataset
152:             # and box_type_3d='Depth' in sunrgbd and scannet dataset.
153:             box_type_3d='LiDAR',
154:             backend_args=backend_args)))
155: test_dataloader = dict(
156:     dataset=dict

# Planned config changes for the reduced baseline

The current config uses:
- `nuscenes_infos_train.pkl` for training
- the standard validation/test settings inherited from the base config
- `CBGSDataset` as the training dataset wrapper

For the reduced-budget baseline, I will:
- replace the training annotation file with the 20% subset info file
- keep validation unchanged
- keep the overall model architecture unchanged

This ensures that the main experimental change is the reduced training data, not a different detector design.

# Updating the training annotation file

I now modify the experiment config so that the training dataset uses the reduced 20% subset info file instead of the full training info file.

Only the training annotation file is changed here:
- training uses the reduced subset
- validation remains unchanged

This keeps evaluation comparable while reducing training cost.

In [8]:
from pathlib import Path

with open(experiment_config_path, "r") as f:
    config_text = f.read()

old_text = "ann_file='nuscenes_infos_train.pkl'"
new_text = "ann_file='subsets/nuscenes_infos_train_20pct.pkl'"

if new_text in config_text:
    print("Training ann_file already points to the 20% subset.")
elif old_text in config_text:
    config_text = config_text.replace(old_text, new_text, 1)
    with open(experiment_config_path, "w") as f:
        f.write(config_text)
    print("Updated training ann_file to the 20% subset.")
else:
    print("Expected training ann_file pattern not found. No change was made.")

print("\nCurrent experiment config:")
print(experiment_config_path)

Training ann_file already points to the 20% subset.

Current experiment config:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/lidar_baseline_nuscenes_20pct.py


# Verifying the training annotation update

After editing the config, I verify that the training dataloader now points to the reduced subset info file.

This is an important check because the experiment depends on using the intended reduced training set.

In [9]:
with open(experiment_config_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "ann_file" in line or "train_dataloader" in line:
        print(f"{i+1:03d}: {line.rstrip()}")

134: train_dataloader = dict(
145:             ann_file='subsets/nuscenes_infos_train_20pct.pkl',


# Training data configuration result

The experiment config now uses the reduced training subset:

`subsets/nuscenes_infos_train_20pct.pkl`

This means:
- training runs on the 20% scene-level subset
- validation remains on the standard validation split
- the main detector architecture stays unchanged

# Inspecting training budget settings

I now inspect the key runtime parameters that affect computation cost:

- batch size
- number of workers
- validation interval
- training schedule (epochs)

These parameters determine how long training will take on the available hardware.

In [10]:
keywords = [
    "batch_size",
    "num_workers",
    "train_cfg",
    "val_interval",
    "max_epochs",
    "train_dataloader",
]

for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        print(f"{i+1:03d}: {line.rstrip()}")

024:     train_cfg=dict(pts=dict(point_cloud_range=point_cloud_range)),
134: train_dataloader = dict(
136:     batch_size=4,
137:     num_workers=16,
160: train_cfg = dict(val_interval=20)
164: train_cfg = dict(by_epoch=True, max_epochs=5, val_interval=1)


# Current training budget settings

The current experiment config uses:

- batch size: 4
- number of workers: 4
- validation interval: 20

These settings are conservative and close to a standard baseline configuration.

For a reduced-budget experiment on the 20% subset, these settings may later be adjusted to:
- improve hardware utilization
- reduce wall-clock training time
- keep the experiment feasible within the available compute budget

# Inspecting the training schedule

The experiment config inherits its training schedule from a base file:

`../_base_/schedules/cyclic-20e.py`

I now inspect this file to determine:
- the total number of epochs
- the learning rate schedule
- whether it needs to be reduced for the subset experiment

In [11]:
schedule_path = MMDET3D_ROOT / "configs" / "_base_" / "schedules" / "cyclic-20e.py"

print("Schedule path:", schedule_path)
print("Exists:", schedule_path.exists())

with open(schedule_path, "r") as f:
    schedule_lines = f.readlines()

print("\nFirst 60 lines of the schedule:\n")
for i, line in enumerate(schedule_lines[:60]):
    print(f"{i+1:03d}: {line.rstrip()}")

Schedule path: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/_base_/schedules/cyclic-20e.py
Exists: True

First 60 lines of the schedule:

001: # For nuScenes dataset, we usually evaluate the model at the end of training.
002: # Since the models are trained by 24 epochs by default, we set evaluation
003: # interval to be 20. Please change the interval accordingly if you do not
004: # use a default schedule.
005: # optimizer
006: lr = 1e-4
007: # This schedule is mainly used by models on nuScenes dataset
008: # max_norm=10 is better for SECOND
009: optim_wrapper = dict(
010:     type='OptimWrapper',
011:     optimizer=dict(type='AdamW', lr=lr, weight_decay=0.01),
012:     clip_grad=dict(max_norm=35, norm_type=2))
013: # learning rate
014: param_scheduler = [
015:     # learning rate scheduler
016:     # During the first 8 epochs, learning rate increases from 0 to lr * 10
017:     # during the next 12 epochs, learning rate decreases from lr

# Inherited training schedule

The baseline config inherits the `cyclic-20e.py` schedule.

Key settings:
- optimizer: AdamW
- learning rate: `1e-4`
- training mode: epoch-based
- maximum epochs: 20
- validation interval: 20

This is a standard nuScenes training schedule, but it may be more expensive than necessary for a reduced 20% subset experiment.

# Defining a reduced-budget training schedule

The default schedule uses:
- 20 epochs
- full validation only at the end

For a reduced 20% subset experiment, I adopt a lighter schedule to:
- reduce wall-clock training time
- allow multiple experiments (baseline, fusion, ablations)
- still obtain meaningful results

## Selected strategy

I use:
- **max_epochs = 1** → fast baseline run
- **val_interval = 1** → evaluate every epoch

This configuration is suitable for:
- debugging the pipeline
- obtaining a first baseline result quickly
- scaling up later if needed

In [12]:
with open(experiment_config_path, "r") as f:
    config_text = f.read()

schedule_override = """
# === Reduced budget schedule ===
train_cfg = dict(by_epoch=True, max_epochs=1, val_interval=1)
"""

if "train_cfg = dict(by_epoch=True, max_epochs=1, val_interval=1)" in config_text:
    print("Reduced schedule already applied.")
else:
    with open(experiment_config_path, "a") as f:
        f.write("\n" + schedule_override)
    print("Appended reduced training schedule.")

print("\nUpdated config:", experiment_config_path)

Applied reduced training schedule.

Updated config: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/lidar_baseline_nuscenes_20pct.py


# Verifying the reduced schedule override

After editing the config, I verify that the experiment file now contains a reduced-budget training schedule override.

This confirms that the inherited 20-epoch schedule has been replaced for this experiment.

In [13]:
with open(experiment_config_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "train_cfg" in line or "max_epochs" in line or "val_interval" in line:
        print(f"{i+1:03d}: {line.rstrip()}")

024:     train_cfg=dict(pts=dict(point_cloud_range=point_cloud_range)),
160: train_cfg = dict(val_interval=20)
164: train_cfg = dict(by_epoch=True, max_epochs=5, val_interval=1)
168: train_cfg = dict(by_epoch=True, max_epochs=1, val_interval=1)


# Reduced-budget schedule result

The experiment config now contains an explicit reduced-budget training override:

- `max_epochs = 1`
- `val_interval = 1`

This override replaces the inherited 20-epoch schedule for the current experiment and is appropriate for:
- a first baseline run
- pipeline validation
- fast turnaround on the reduced 20% training subset

# Optimizing batch size and data loading

The current config uses:
- batch size = 4
- num_workers = 4

These values are conservative.

## Goal
Improve data-loading throughput while keeping training stable for sparse LiDAR detection.

## Strategy
- keep the per-GPU batch size conservative to avoid sparse CUDA runtime failures
- increase the number of workers to improve data loading speed

## Selected values for this experiment
- batch size = 4
- num_workers = 16

These values are reasonable for:
- CenterPoint on nuScenes
- a single H100 GPU
- a stable first baseline run on the reduced subset

In [14]:
with open(experiment_config_path, "r") as f:
    config_text = f.read()

config_text_new = config_text

config_text_new = config_text_new.replace("batch_size=4", "batch_size=4")
config_text_new = config_text_new.replace("num_workers=4", "num_workers=16")

if config_text_new != config_text:
    with open(experiment_config_path, "w") as f:
        f.write(config_text_new)
    print("Updated batch size and num_workers.")
else:
    print("No changes made (values may already be updated).")

print("\nConfig path:", experiment_config_path)

No changes made (values may already be updated).

Config path: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/lidar_baseline_nuscenes_20pct.py


# Verifying data-loading updates

After editing the config, I verify that the runtime settings now use the intended:
- batch size
- number of workers

This confirms that the reduced-budget baseline is configured for higher throughput on the available hardware.

In [15]:
with open(experiment_config_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "batch_size" in line or "num_workers" in line:
        print(f"{i+1:03d}: {line.rstrip()}")

136:     batch_size=4,
137:     num_workers=16,


# Training command

The baseline model can be trained using the following command:

```bash
cd /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d

conda activate py38_mmdet3d

python tools/train.py configs/my_experiments/lidar_baseline_nuscenes_20pct.py

# Main training schedule after the smoke test

The 1-epoch configuration is used only as a smoke test to verify that:
- the subset dataset loads correctly
- the model trains without runtime errors
- logging, checkpoints, and validation work as expected

For the actual reduced-budget baseline, I use:

- max_epochs = 5
- val_interval = 1

## Rationale
This is a reasonable compromise between:
- computational cost
- experiment turnaround time
- obtaining a meaningful baseline result on the 20% training subset

A 5-epoch run is substantially stronger than a smoke test while still remaining feasible within the available H100 session budget.

In [21]:
with open(experiment_config_path, "r") as f:
    config_text = f.read()

import re

# Remove previously appended reduced-budget schedule blocks
config_text = re.sub(
    r"\n+# === Reduced budget schedule ===\ntrain_cfg = dict\(by_epoch=True, max_epochs=\d+, val_interval=\d+\)\n?",
    "",
    config_text
)

# Append exactly one final schedule override
config_text += "\n\n# === Reduced budget schedule ===\ntrain_cfg = dict(by_epoch=True, max_epochs=5, val_interval=1)\n"

with open(experiment_config_path, "w") as f:
    f.write(config_text)

print("Cleaned old reduced-budget schedule overrides.")
print("Applied a single final schedule: max_epochs=5, val_interval=1")
print("\nUpdated config:", experiment_config_path)

Cleaned old reduced-budget schedule overrides.
Applied a single final schedule: max_epochs=5, val_interval=1

Updated config: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/configs/my_experiments/lidar_baseline_nuscenes_20pct.py


In [22]:
with open(experiment_config_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "train_cfg = dict(by_epoch=True" in line or "max_epochs" in line or "val_interval" in line:
        print(f"{i+1:03d}: {line.rstrip()}")

160: train_cfg = dict(val_interval=20)
163: train_cfg = dict(by_epoch=True, max_epochs=5, val_interval=1)


## Running the baseline training with SLURM

For long training runs on UBELIX, it is safer to submit the experiment through SLURM instead of launching it from a fragile interactive terminal.

### Why this is useful
Using SLURM makes the training run independent of the local computer state.

This means:
- the job keeps running even if the laptop sleeps
- the job is not interrupted by SSH disconnection
- resource requests are explicit and reproducible

### Strategy
I create a small SLURM submission script that:
- requests one GPU
- sets a 12-hour wall-time limit
- activates the correct conda environment
- launches the MMDetection3D training command
- can resume from the latest checkpoint if needed

This is important because the main baseline training may take longer than a single 12-hour session.

# Defining the SLURM script location

I save the SLURM submission script inside the project directory so that:
- the training procedure is reproducible
- the script can be reused for later experiments
- the execution setup is documented together with the project

In [23]:
slurm_dir = PROJECT_ROOT / "slurm"
slurm_dir.mkdir(parents=True, exist_ok=True)

slurm_script_path = slurm_dir / "train_lidar_baseline_nuscenes_20pct.slurm"

print("SLURM script path:", slurm_script_path)

SLURM script path: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/train_lidar_baseline_nuscenes_20pct.slurm


# Creating a resume-ready SLURM submission script

The script below is designed for a single-GPU UBELIX training run with a 12-hour wall-time limit.

UBELIX requires the GPU type to be specified explicitly in the SLURM request.
For this experiment, I request one H100 GPU.

The script supports both:
- a fresh training run if no checkpoint is present
- a resumed training run if a previous checkpoint already exists

In [28]:
work_dir = MMDET3D_ROOT / "work_dirs" / "lidar_baseline_nuscenes_20pct"

slurm_script = f"""#!/bin/bash
#SBATCH --job-name=lidar_20pct
#SBATCH --output={PROJECT_ROOT}/slurm/lidar_20pct_%j.out
#SBATCH --error={PROJECT_ROOT}/slurm/lidar_20pct_%j.err
#SBATCH --time=12:00:00
#SBATCH --nodes=1
#SBATCH --gpus-per-node=h100:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G

source ~/.bashrc
conda activate py38_mmdet3d

cd {MMDET3D_ROOT}
echo "Working directory: $(pwd)"

CHECKPOINT_PATH="{work_dir}/latest.pth"

if [ -f "$CHECKPOINT_PATH" ]; then
    echo "Resuming training from $CHECKPOINT_PATH"
    python tools/train.py configs/my_experiments/lidar_baseline_nuscenes_20pct.py \\
        --resume "$CHECKPOINT_PATH"
else
    echo "No checkpoint found. Starting fresh training run."
    python tools/train.py configs/my_experiments/lidar_baseline_nuscenes_20pct.py
fi
"""

with open(slurm_script_path, "w") as f:
    f.write(slurm_script)

print("Saved SLURM script to:")
print(slurm_script_path)

Saved SLURM script to:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/train_lidar_baseline_nuscenes_20pct.slurm


# Verifying the generated SLURM script

Before submitting the job, I inspect the generated script to confirm that:
- the requested resources are correct
- the environment activation is correct
- the training command points to the correct experiment config
- the script checks for an existing checkpoint and resumes automatically when available

In [29]:
with open(slurm_script_path, "r") as f:
    slurm_lines = f.readlines()

for i, line in enumerate(slurm_lines):
    print(f"{i+1:02d}: {line.rstrip()}")

01: #!/bin/bash
02: #SBATCH --job-name=lidar_20pct
03: #SBATCH --output=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/lidar_20pct_%j.out
04: #SBATCH --error=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/lidar_20pct_%j.err
05: #SBATCH --time=12:00:00
06: #SBATCH --nodes=1
07: #SBATCH --gpus-per-node=h100:1
08: #SBATCH --cpus-per-task=16
09: #SBATCH --mem=64G
10: 
11: source ~/.bashrc
12: conda activate py38_mmdet3d
13: 
14: cd /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d
15: echo "Working directory: $(pwd)"
16: 
17: CHECKPOINT_PATH="/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/work_dirs/lidar_baseline_nuscenes_20pct/latest.pth"
18: 
19: if [ -f "$CHECKPOINT_PATH" ]; then
20:     echo "Resuming training from $CHECKPOINT_PATH"
21:     python tools/train.py configs/my_experiments/lidar_baseline_nuscenes_20pct.py \
22:         --resume "$CHECKPOINT_PATH"
23: else
2

# Enabling checkpoint saving for safe resume

## Goal

The experiment config should explicitly save checkpoints so training can be resumed safely after an interruption.

## Why this matters

This experiment inherits most runtime settings from the base config `default_runtime.py`.  
If checkpoint saving is not made explicit in the experiment config, it is harder to verify whether `.pth` files will be written as expected.

Adding an explicit checkpoint hook makes the behavior clear and reproducible.

## Strategy

We append a `default_hooks` block to the experiment config so that:

- a checkpoint is saved every epoch
- the latest checkpoint is always kept
- only a small number of checkpoints are retained

This is a good setup for reduced-budget training on HPC.

In [ ]:
with open(experiment_config_path, "r") as f:
    config_text = f.read()

config_text_new = config_text

checkpoint_block = """

# === Checkpoint saving for safe resume ===
default_hooks = dict(
    checkpoint=dict(
        type='CheckpointHook',
        interval=1,
        save_last=True,
        max_keep_ckpts=3))
"""

if "default_hooks = dict(" not in config_text_new:
    config_text_new = config_text_new.rstrip() + checkpoint_block + "\n"
    with open(experiment_config_path, "w") as f:
        f.write(config_text_new)
    print("Added explicit checkpoint hook to the experiment config.")
else:
    print("default_hooks already exists in the config. Please inspect it before modifying.")

print("\nConfig path:", experiment_config_path)

### Verification

The next cell checks whether the checkpoint settings are now explicitly present in the experiment config.

In [1]:
with open(experiment_config_path, "r") as f:
    updated_text = f.read()

for key in ["default_hooks", "CheckpointHook", "interval=1", "save_last=True", "max_keep_ckpts=3"]:
    print(f"{key}: {'FOUND' if key in updated_text else 'MISSING'}")

NameError: name 'experiment_config_path' is not defined

# Expected result

After this update, completed epochs should produce checkpoint files such as:

- `epoch_1.pth`
- `latest.pth`

These files are usually written in the main experiment directory under `work_dirs`, not inside the timestamped log subfolder.

# SLURM submission and resume behavior

The training job can be submitted with:

```bash
cd /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
sbatch slurm/train_lidar_baseline_nuscenes_20pct.slurm
```

### Monitoring
Useful commands after submission:

```bash
squeue -u ae04q066
```

```bash
tail -f /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/slurm/lidar_20pct_<JOBID>.out
```

### Resume behavior
The SLURM script automatically checks whether the following checkpoint exists:

```bash
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/work_dirs/lidar_baseline_nuscenes_20pct/latest.pth
```

- if `latest.pth` exists, training resumes from that checkpoint
- if `latest.pth` does not exist, training starts from scratch

This makes it possible to continue training across multiple 12-hour SLURM jobs without manually editing the command each time.

### Cancel job
```bash
scancel <JOBID>
```


### Benefit
This execution method is robust to:
- laptop sleep
- SSH disconnects
- unstable local network connections
- wall-time limits requiring multiple job submissions

# Note on checkpoint-based continuation

MMDetection3D saves checkpoints in the experiment work directory.

Using `latest.pth` is convenient because:
- it always points to the most recent checkpoint
- it avoids manually tracking the latest epoch number
- it simplifies continuation across multiple SLURM jobs

This is especially useful for experiments that may take longer than one 12-hour allocation.